# 4. Develop, compare and build incident queues

Label-free thresholds are saved before evaluation reads development truth. Selection can fail closed.

In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "pyproject.toml").exists()
)
sys.path.insert(0, str(ROOT / "src"))
from telco_anomaly.pipeline import runtime

POLICY = runtime(ROOT / "configs/pipeline.yml")
PACK = Path(POLICY["pack"])
RUN = Path(POLICY["run"])


In [ ]:
from telco_anomaly.pipeline import develop, frozen_run

if not RUN.exists():
    comparison = develop(POLICY)
else:
    receipt = frozen_run(RUN)
    assert receipt["policy"] == POLICY, "Settings changed: use a new run path"
    comparison = pd.read_csv(RUN / "comparison.csv")
display(comparison.T)
print((RUN / "selection_status.json").read_text())
print((RUN / "model_card.md").read_text())

In [ ]:
queue_path = RUN / "full_queue.csv"
if queue_path.exists():
    display(pd.read_csv(queue_path).head(20))
display(pd.DataFrame(json.loads((RUN / "label_free_choices.json").read_text())))

Queues consolidate alerts by port and quiet period, map reviewed power events, and rank candidate topology footprints with explicit structural ambiguity. The ranking probabilities are uncalibrated and conditional on a simplified single-fault model. Verification budgets conservatively count entity alerts; development metrics count consolidated cases. Sparse-tail extrapolation is disabled. Low-support results remain descriptive, and unsuccessful selection does not create a deployable model.